In [ ]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("=" * 80)
print("[STEP 0] Notebook parameters received")
print("=" * 80)
print(f"[PARAM] landing_timestamp = '{landing_timestamp}'")
print(f"[PARAM] table_name        = '{table_name}'")
print(f"[PARAM] merge_keys        = '{merge_keys}'")
print(f"[PARAM] load_type         = '{load_type}' (empty means FULL)")

if not table_name:
    print("[WARNING] table_name is empty - source/target table names will be invalid")
if not landing_timestamp:
    print("[WARNING] landing_timestamp is empty - source filter will return no rows")
print("[STEP 0 COMPLETE] Parameters loaded")


In [ ]:
print("=" * 80)
print("[STEP 1] Reading source bronze table")
print("=" * 80)
source_table = f"adb_healthy.bronze.{table_name}"
print(f"[SOURCE TABLE] {source_table}")
print(f"[SOURCE FILTER] landing_timestamp = '{landing_timestamp}'")
source_query = f"SELECT * FROM adb_healthy.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'"
print(f"[SOURCE QUERY]\n{source_query}")

bronze_df = spark.read.table(source_table).filter(
    f"landing_timestamp='{landing_timestamp}'"
)

print(
    f"[STEP 1 COMPLETE] Source DataFrame prepared for table '{table_name}' with landing_timestamp '{landing_timestamp}'"
)
print(f"[SOURCE COLUMNS] ({len(bronze_df.columns)}) {bronze_df.columns}")
print("[SOURCE SCHEMA]")
bronze_df.printSchema()

source_row_count = bronze_df.count()
print(f"[SOURCE ROW COUNT] {source_row_count} row(s) found for landing_timestamp '{landing_timestamp}'")
if source_row_count == 0:
    print("[WARNING] No source rows found - the write step will not change any data")

In [ ]:
from pyspark.sql import functions as F

print("=" * 80)
print("[STEP 2] Preparing load metadata")
print("=" * 80)
target_table = f"adb_healthy.silver.{table_name}"
normalized_load_type = (load_type or "FULL").strip().upper()
merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]

print(f"[TARGET TABLE] {target_table}")
print(f"[RAW LOAD TYPE] '{load_type}'")
print(f"[NORMALIZED LOAD TYPE] {normalized_load_type}")
print(f"[MERGE KEYS] {merge_key_list}")
print(f"[TARGET EXISTS] {spark.catalog.tableExists(target_table)}")
if normalized_load_type != "MERGE" and merge_key_list:
    print(f"[INFO] merge_keys are ignored for load_type '{normalized_load_type}'")

if normalized_load_type == "MERGE":
    print("[MERGE VALIDATION] Validating merge keys against source columns")
    if not merge_key_list:
        raise ValueError("merge_keys must be provided when load_type is 'MERGE'.")

    missing_merge_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_merge_keys:
        raise ValueError(f"merge_keys not found in source data: {missing_merge_keys}")

    print("[MERGE VALIDATION COMPLETE] All merge keys are present in source data")

    duplicate_key_count = (
        bronze_df.groupBy(*merge_key_list).count().filter("count > 1").count()
    )
    print(f"[MERGE VALIDATION] Duplicate merge key groups in source: {duplicate_key_count}")
    if duplicate_key_count > 0:
        print("[WARNING] Source has duplicate merge keys - MERGE will fail if they match the same target row")

print("[AUDIT COLUMNS] Adding insert_timestamp and update_timestamp = current_timestamp()")

updated_df = (
    bronze_df.withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)

print("[STEP 2 COMPLETE] Added insert_timestamp and update_timestamp columns")
print(f"[UPDATED COLUMNS] {updated_df.columns}")

In [ ]:
print("=" * 80)
print(f"[STEP 3] Starting write to target table: {target_table}")
print("=" * 80)

def print_last_operation_metrics(table):
    """Print the operation and metrics of the latest Delta commit on the table."""
    history_query = f"DESCRIBE HISTORY {table} LIMIT 1"
    print(f"[HISTORY QUERY] {history_query}")
    last_op = spark.sql(history_query).select("version", "operation", "operationMetrics").first()
    print(f"[DELTA VERSION] {last_op['version']}")
    print(f"[DELTA OPERATION] {last_op['operation']}")
    for metric_name, metric_value in sorted((last_op["operationMetrics"] or {}).items()):
        print(f"    {metric_name}: {metric_value}")

    count_query = f"SELECT COUNT(*) AS cnt FROM {table}"
    print(f"[COUNT QUERY] {count_query}")
    print(f"[TARGET ROW COUNT AFTER WRITE] {spark.sql(count_query).first()['cnt']}")

# SQL equivalent of updated_df (audit columns replace any existing ones from bronze)
audit_columns = {"insert_timestamp", "update_timestamp"}
select_list = ",\n       ".join(
    [f"`{c}`" for c in bronze_df.columns if c not in audit_columns]
    + ["current_timestamp() AS insert_timestamp", "current_timestamp() AS update_timestamp"]
)
equivalent_select = (
    f"SELECT {select_list}\nFROM {source_table}\nWHERE landing_timestamp = '{landing_timestamp}'"
)

if normalized_load_type == "FULL":
    print("[WRITE MODE] FULL")
    print(f"[WRITE ACTION] Overwriting target table {target_table}")
    print("[EQUIVALENT SQL]")
    print(f"CREATE OR REPLACE TABLE {target_table} AS\n{equivalent_select}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print_last_operation_metrics(target_table)
    print("[STEP 3 COMPLETE] FULL overwrite finished successfully")
elif normalized_load_type == "APPEND":
    print("[WRITE MODE] APPEND")
    print(f"[WRITE ACTION] Appending records into target table {target_table}")
    print("[EQUIVALENT SQL]")
    print(f"INSERT INTO {target_table}\n{equivalent_select}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print_last_operation_metrics(target_table)
    print("[STEP 3 COMPLETE] APPEND finished successfully")
elif normalized_load_type == "MERGE":
    print("[WRITE MODE] MERGE")
    target_exists = spark.catalog.tableExists(target_table)
    print(f"[TARGET EXISTS] {target_exists}")

    if not target_exists:
        print("[MERGE FALLBACK] Target table does not exist. Creating target table with current dataset.")
        print("[EQUIVALENT SQL]")
        print(f"CREATE OR REPLACE TABLE {target_table} AS\n{equivalent_select}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print_last_operation_metrics(target_table)
        print("[STEP 3 COMPLETE] Target table created for initial MERGE load")
    else:
        source_view = "silver_merge_source"
        updated_df.createOrReplaceTempView(source_view)
        print(f"[TEMP VIEW] Created temp view '{source_view}' for merge source")
        print(f"[MERGE CONDITION] matching on keys {merge_key_list}")
        print("[MERGE BEHAVIOR] Matched rows -> UPDATE all columns except insert_timestamp; unmatched rows -> INSERT")

        merge_condition = " AND ".join(
            [f"target.`{key}` = source.`{key}`" for key in merge_key_list]
        )
        update_assignments = ", ".join(
            [
                f"target.`{column}` = source.`{column}`"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )
        insert_columns = ", ".join([f"`{column}`" for column in updated_df.columns])
        insert_values = ", ".join([f"source.`{column}`" for column in updated_df.columns])

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET {update_assignments}
            WHEN NOT MATCHED THEN INSERT ({insert_columns})
            VALUES ({insert_values})
        """.strip()

        print("[MERGE SQL]")
        print(merge_sql)

        print("[MERGE EXECUTION] Running MERGE statement...")
        merge_result = spark.sql(merge_sql).first()
        if merge_result is not None:
            print(f"[MERGE RESULT] {merge_result.asDict()}")
        print_last_operation_metrics(target_table)
        print("[STEP 3 COMPLETE] MERGE finished successfully")
else:
    raise ValueError("load_type must be one of: FULL, APPEND, MERGE")

In [ ]:
print("=" * 80)
print("[STEP 4] Computing processed record count")
print("=" * 80)
record_count = updated_df.count()
print(f"[STEP 4 COMPLETE] Processed record count: {record_count}")
print(f"[SUMMARY] table={table_name} | load_type={normalized_load_type} | landing_timestamp={landing_timestamp} | records={record_count}")
print(f"[EXIT] Returning '{record_count}' to the caller via dbutils.notebook.exit")
dbutils.notebook.exit(str(record_count))
